# 02 — Stage 3 contact sheet
ip_scale × controlnet_scale 격자를 한 face × 한 species 별로 본다.

In [ ]:
import sys, os
from pathlib import Path
REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO)); os.chdir(REPO)

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

In [ ]:
runs = sorted(Path('experiments/runs').glob('stage3_*'))
RUN = runs[-1]
print('run:', RUN)
df = pd.read_parquet(RUN / 'metadata.parquet')
df = df[df['image_path'] != ''].copy()
print(sorted(df['ip_scale'].unique()), sorted(df['controlnet_scale'].unique()))

In [ ]:
SEED = 0
ip_scales = sorted(df['ip_scale'].unique())
cn_scales = sorted(df['controlnet_scale'].unique())

for fid in sorted(df['face_id'].unique()):
    for sp in sorted(df['species'].unique()):
        sub = df[(df['face_id']==fid) & (df['species']==sp) & (df['seed']==SEED)]
        if sub.empty: continue
        fig, axes = plt.subplots(len(ip_scales), len(cn_scales), figsize=(2.4*len(cn_scales), 2.4*len(ip_scales)))
        if len(ip_scales)==1: axes = [axes]
        for r, ip in enumerate(ip_scales):
            for c, cn in enumerate(cn_scales):
                ax = axes[r][c] if len(ip_scales)>1 else axes[c]
                ax.axis('off')
                row = sub[(sub['ip_scale']==ip) & (sub['controlnet_scale']==cn)]
                if not row.empty: ax.imshow(Image.open(row.iloc[0]['image_path']))
                if r == 0: ax.set_title(f'cn={cn:.1f}')
                if c == 0: ax.set_ylabel(f'ip={ip:.1f}')
        fig.suptitle(f'{fid} / {sp}'); plt.tight_layout(); plt.show()